In [2]:
!pip install -q xgboost

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.utils.class_weight import compute_sample_weight

from xgboost import XGBClassifier

import joblib

print("Libraries imported successfully")

Libraries imported successfully


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
engine_path = "/content/drive/MyDrive/AutoSenseAI/data/engine_data.csv"

engine_df = pd.read_csv(engine_path)

engine_df.head()

,Engine rpm,Lub oil pressure,Fuel pressure,Coolant pressure,lub oil temp,Coolant temp,Engine Condition
0,700,2.493592,11.790927,3.178981,84.144163,81.632187,1
1,876,2.941606,16.193866,2.464504,77.640934,82.445724,0
2,520,2.961746,6.553147,1.064347,77.752266,79.645777,1
3,473,3.707835,19.510172,3.727455,74.129907,71.774629,1
4,619,5.672919,15.738871,2.052251,78.396989,87.000225,0


In [6]:
X = engine_df.drop(
    columns=["Engine Condition"]
)

y = engine_df["Engine Condition"]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [8]:
engine_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,

    objective="binary:logistic",
    eval_metric="logloss",

    random_state=42,
    n_jobs=-1
)

In [9]:
engine_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

In [10]:
engine_model.fit(
    X_train,
    y_train,
    sample_weight=engine_weights
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=-1,
              num_parallel_tree=None, ...)

In [11]:
y_pred = engine_model.predict(X_test)

print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.51      0.65      0.57      1444
           1       0.75      0.63      0.69      2463

    accuracy                           0.64      3907
   macro avg       0.63      0.64      0.63      3907
weighted avg       0.66      0.64      0.64      3907



In [12]:
print(
    "Accuracy:",
    accuracy_score(y_test, y_pred)
)

print(
    "Macro F1:",
    f1_score(
        y_test,
        y_pred,
        average="macro"
    )
)

Accuracy: 0.6380854875863834
Macro F1: 0.6289514053441226


In [14]:
import os
import joblib

# Folder where the trained model will be saved
model_dir = "/content/drive/MyDrive/AutoSenseAI/models"

# Create the folder if it does not already exist
os.makedirs(model_dir, exist_ok=True)

# Full file path
model_path = os.path.join(
    model_dir,
    "engine_condition_xgboost.pkl"
)

# Save trained XGBoost model
joblib.dump(engine_model, model_path)

print("Model saved successfully!")
print("Saved location:", model_path)

Model saved successfully!
Saved location: /content/drive/MyDrive/AutoSenseAI/models/engine_condition_xgboost.pkl


In [15]:
print(os.path.exists(model_path))

True


In [16]:
print(os.listdir(model_dir))

['vehicle_maintenance_xgboost.pkl', 'engine_condition_xgboost.pkl']


In [17]:
joblib.dump(
    engine_model,
    model_dir + "/engine_condition_xgboost.pkl"
)

['/content/drive/MyDrive/AutoSenseAI/models/engine_condition_xgboost.pkl']